# ESCM 2.0 — multi-pixel stress test of counter-free inhibition

**Question.** The two models generated exactly the same output spikes across all 11 single-pixel/neighbor protocols in notebook 07, despite small differences in inhibitory state. Does the counter-free mechanism still preserve useful behavior when *many adjacent pixels* are active?

The baseline and candidate retain the same **corrected** neuronal reset, Brian2 custom-event scheduling, self-excitation and explicit recurrent pathway order. Only the inhibitory update changes:

\[
\begin{aligned}
\text{Baseline: }&C_i\leftarrow C_i+1,\quad
  \mathrm{Inh}_i\leftarrow P_iW_kC_i;\\
\text{Candidate: }&\mathrm{Inh}_i\leftarrow\mathrm{Inh}_i+P_iW_k.
\end{aligned}
\]

We intentionally use a **9 × 7 grid** and compare \(k=2\) with \(k=8\). Unlike the 5 × 5 network with \(k=8\), the smaller radius is not all-to-all. The resulting comparison tests spatial structure and the mutual inhibition that caused the earlier five-pixel edge to extinguish.

The paper does not specify these particular input timings; they are controlled stress tests, not a reproduction claim.

## 1. Define moving-edge protocols

A vertical edge of five pixels traverses five successive columns. The protocol varies the time between columns and, separately, whether all five pixels in a column spike simultaneously or in a staggered sequence.

We simulate for 60 ms, including a silent interval after the last input. The desired qualitative behavior is that the newest edge is represented without excessive surviving activity from older positions. We do **not** require exact equality between different inhibitory laws.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from brian2 import *
from tqdm.auto import tqdm

prefs.codegen.target = "numpy"
width, height = 9, 7
N = width * height
rows = np.arange(1, 6)
columns = np.arange(2, 7)
dt_ms = 0.5
end_ms = 60.0
tau_model = 0.2 * ms
vt_model, vr_model = 0.1, 0.0
beta_model, Wi_model, Wk_model = 0.5, 6.0, -3.0

protocols = {}
for interval in (1.0, 4.0, 8.0):
    for stagger in (0.0, 0.5):
        ids, times = [], []
        for step, col in enumerate(columns):
            for row_index, row in enumerate(rows):
                ids.append(int(row * width + col))
                times.append(1.0 + step * interval + row_index * stagger)
        name = f"edge_{interval:g}ms_" + ("staggered" if stagger else "simultaneous")
        protocols[name] = (np.asarray(ids, dtype=int), np.asarray(times))
for name, (ids, times) in protocols.items():
    ticks = np.rint(times / dt_ms).astype(int)
    assert np.allclose(ticks * dt_ms, times)
    assert len(set(zip(ticks, ids))) == len(ids)
    assert times.max() < end_ms - 10
print(f"{len(protocols)} edge protocols, {N} neurons, time step {dt_ms} ms")
print("Rows:", rows.tolist(), "Columns:", columns.tolist())

## 2. Run both inhibitory mechanisms on identical inputs

The state equation remains \(\tau\dot v=-v+\mathrm{ExtIn}+P(\mathrm{Exc}+\mathrm{Inh})\), with exponential decay of the three input traces.

Both variants preserve the legacy persistence rules. The counter-free candidate retains a declared integer counter for monitoring compatibility, but does not update or read it.

The code uses explicit group namespaces to avoid ambiguous-name warnings. A progress bar reports the simulations as they complete.

In [ ]:
def run_model(mode, k, ids, times):
    start_scope()
    defaultclock.dt = dt_ms * ms
    params = dict(tau_model=tau_model, vt_model=vt_model, vr_model=vr_model,
                  beta_model=beta_model, Wi_model=Wi_model, Wk_model=Wk_model)
    camera = SpikeGeneratorGroup(N, ids, times * ms)
    eqs = """
    dv/dt = (-v + ExtIn + P*(Exc + Inh))/tau_model : 1
    dExtIn/dt = -ExtIn/tau_model : 1
    dExc/dt = -Exc/tau_model : 1
    dInh/dt = -Inh/tau_model : 1
    incoming_spikes : integer
    P : integer
    """
    neurons = NeuronGroup(
        N, eqs, threshold="v > vt_model",
        reset="v = vr_model; incoming_spikes = 0",
        refractory=0 * ms,
        events={"P_ON": "v > vt_model",
                "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= vt_model)"},
        method="exact", namespace=params,
    )
    neurons.set_event_schedule("P_ON", when="after_thresholds")
    neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
    neurons.set_event_schedule("P_OFF", when="groups")
    neurons.run_on_event("P_OFF", "P = 0", when="groups")

    feed = Synapses(camera, neurons, on_pre="ExtIn_post = beta_model", namespace=params)
    feed.connect(j="i")

    if mode == "baseline":
        on_pre = {"pre": "incoming_spikes_post += 1; Exc_pre = Wi_model",
                  "pre_2": "Inh_post = P_post * Wk_model * incoming_spikes_post"}
    else:
        on_pre = {"pre": "Exc_pre = Wi_model",
                  "pre_2": "Inh_post += P_post * Wk_model"}
    recurrent = Synapses(neurons, neurons, on_pre=on_pre, namespace=params)
    recurrent.pre.order = 0
    recurrent.pre_2.order = 1
    src = np.repeat(np.arange(N), N)
    dst = np.tile(np.arange(N), N)
    use = (src != dst) & (abs(src % width - dst % width) <= k) & (
        abs(src // width - dst // width) <= k)
    recurrent.connect(i=src[use], j=dst[use])

    spikes = SpikeMonitor(neurons)
    net = Network(camera, neurons, feed, recurrent, spikes)
    net.run(end_ms * ms)
    return np.rint(np.asarray(spikes.t / ms) / dt_ms).astype(int), np.asarray(spikes.i, dtype=int)

results = {}
jobs = [(k, name, mode) for k in (2, 8) for name in protocols
        for mode in ("baseline", "candidate")]
for k, name, mode in tqdm(jobs, desc="Edge comparisons"):
    ids, times = protocols[name]
    results[k, name, mode] = run_model(mode, k, ids, times)

## 3. Compare exact spikes and persistence at the final edge

We test exact equality of \((\mathrm{tick},\mathrm{neuron})\) pairs, but interpret differences against the two behavioral objectives:

**Retention:** do output spikes survive after the final input? **Replacement:** do the final-column pixels dominate older columns during the silent window?

The following summary reports spike counts within **20 ms after the last event**, not an arbitrary fixed interval that might overlap an unfinished stimulus.

In [ ]:
rows_out = []
for k in (2, 8):
    for name, (ids, times) in protocols.items():
        tb, ib = results[k, name, "baseline"]
        tc, ic = results[k, name, "candidate"]
        same = np.array_equal(np.column_stack((tb, ib)), np.column_stack((tc, ic)))
        cutoff = times.max() + 2 * dt_ms
        for mode, ticks, neuron_ids in (("baseline", tb, ib), ("candidate", tc, ic)):
            active = ticks * dt_ms >= cutoff
            n_active = np.count_nonzero(active)
            at_final = np.count_nonzero(active & np.isin(neuron_ids % width, columns[-1]))
            at_previous = np.count_nonzero(active & np.isin(neuron_ids % width, columns[:-1]))
            rows_out.append((k, name, mode, len(ticks), n_active, at_final,
                             at_previous, same))
print(f"{'k':>2s} {'Protocol':28s} {'Model':10s} {'Total':>6s} {'Silent':>7s} {'Final':>6s} {'Old':>6s} {'Exact':>6s}")
print("-"*84)
for k, name, mode, total, silent, last, old, same in rows_out:
    print(f"{k:2d} {name:28s} {mode:10s} {total:6d} {silent:7d} {last:6d} {old:6d} {str(same):>6s}")
print("\nProtocols with exact spike-train agreement:")
for k in (2, 8):
    count = sum(r[-1] for r in rows_out if r[0] == k and r[2] == "baseline")
    print(f"  k={k}: {count}/{len(protocols)}")

## 4. Inspect the spatial spike rasters and retention

Select one slow simultaneous-edge stimulus and one faster staggered stimulus. Compare baseline and candidate at both radii. Neuron indices are displayed on the same axes so timing and suppression can be examined without changing frame windows.

In [ ]:
shown = ("edge_4ms_simultaneous", "edge_1ms_staggered")
fig, axes = plt.subplots(4, 2, figsize=(13, 11), sharex=True, sharey=True,
                         constrained_layout=True)
for row, (k, name) in enumerate(((2, shown[0]), (2, shown[1]),
                                (8, shown[0]), (8, shown[1]))):
    for col, mode in enumerate(("baseline", "candidate")):
        ax = axes[row, col]
        ticks, neurons = results[k, name, mode]
        ax.scatter(ticks * dt_ms, neurons, s=10, alpha=0.8)
        input_ids, input_times = protocols[name]
        ax.scatter(input_times, input_ids, marker="x", s=35,
                   color="tab:red", linewidth=1.3)
        ax.axvline(input_times.max(), color="0.3", linestyle="--", linewidth=1)
        ax.set(xlim=(0, end_ms), ylim=(0, N), ylabel=f"k={k}: {name}\nNeuron ID" if col == 0 else "")
        if row == 0:
            ax.set_title(mode)
        if row == 3:
            ax.set_xlabel("Time (ms)")
fig.suptitle("Moving five-pixel edge: baseline vs counter-free inhibition")
plt.show()

## 5. Interpretation and next milestone

A match in all the earlier two-neuron protocols establishes that the counter-free update can preserve their exact output spikes at the tested timestep. This experiment extends the test to **dense spatial interactions**, where simultaneous inhibition and counter history are more consequential.

**Success criterion:** the candidate should reproduce the desirable combination of persistence and suppression, not necessarily identical spikes. The novel update is a *change in the scientific mechanism*, so numerical equality is not required.

**If simultaneous edges extinguish both models**, that is a property of this stimulus and parameter configuration, not evidence that the models are equivalent in general. The next experiment should target independent excitatory and inhibitory decay times, with the current baseline fixed, rather than changing several mechanisms at once.

The code remains an exploratory Brian2 notebook; no reusable software framework is introduced.